In [1]:
import pandas as pd
import numpy as np
import datetime as dt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# 1. Gün ürettiğimiz temiz veriyi okuyoruz
df = pd.read_csv('../data/processed/cleaned_retail.csv')

# InvoiceDate sütununu datetime tipine dönüştürelim
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Toplam Satır: {df.shape[0]}")
df.head()

Toplam Satır: 407664


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00


In [2]:
print("Veri Setindeki İlk Fatura Tarihi:", df['InvoiceDate'].min())
print("Veri Setindeki Son Fatura Tarihi :", df['InvoiceDate'].max())

today_date = df['InvoiceDate'].max() + dt.timedelta(days=2)
print("Analiz Tarihi (Snapshot Date)   :", today_date)

Veri Setindeki İlk Fatura Tarihi: 2009-12-01 07:45:00
Veri Setindeki Son Fatura Tarihi : 2010-12-09 20:01:00
Analiz Tarihi (Snapshot Date)   : 2010-12-11 20:01:00


In [3]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda date: (today_date - date.max()).days,
    'Invoice': lambda num: num.nunique(),
    'TotalPrice': lambda price: price.sum()
})

# Sütun isimlerini standart RFM terimleriyle güncelleyelim
rfm.columns = ['recency', 'frequency', 'monetary']

# Monetary değeri 0'dan büyük olanları filtreleyelim
rfm = rfm[rfm['monetary'] > 0]

print(f"Toplam Müşteri Sayısı: {rfm.shape[0]}")
rfm.head()

Toplam Müşteri Sayısı: 4312


,recency,frequency,monetary
Customer ID,,,
12346,166,11,372.86
12347,4,2,1323.32
12348,75,1,222.16
12349,44,3,2671.14
12351,12,1,300.93


In [4]:
customer_tenure = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda date: (today_date - date.min()).days
}).rename(columns={'InvoiceDate': 'tenure'})

rfm['tenure'] = customer_tenure['tenure']
rfm['avg_order_value'] = rfm['monetary'] / rfm['frequency']

rfm.head()

,recency,frequency,monetary,tenure,avg_order_value
Customer ID,,,,,
12346,166,11,372.86,362,33.90
12347,4,2,1323.32,41,661.66
12348,75,1,222.16,75,222.16
12349,44,3,2671.14,226,890.38
12351,12,1,300.93,12,300.93


In [5]:
rfm.describe([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
recency,4312.00,92.17,96.86,2.00,2.00,4.00,19.00,54.00,137.00,305.00,369.00,375.00
frequency,4312.00,4.46,8.17,1.00,1.00,1.00,1.00,2.00,5.00,13.00,31.00,205.00
monetary,4312.00,2048.24,8914.48,2.95,40.59,110.70,307.99,706.02,1723.14,6237.60,20137.24,349164.35
tenure,4312.00,226.49,118.91,2.00,10.00,26.00,118.00,254.00,330.00,373.00,375.00,375.00
avg_order_value,4312.00,378.32,492.56,2.95,33.91,90.02,182.09,287.37,423.58,920.83,1954.55,11880.84


In [6]:
import os

output_path = os.path.abspath("../data/processed/rfm_features.csv")
rfm.to_csv(output_path)
print(f"RFM özellikleri başarıyla kaydedildi: {output_path}")

RFM özellikleri başarıyla kaydedildi: C:\Users\USER\ecommerce-intelligence\data\processed\rfm_features.csv
